# Operating Systems — A 30,000-Foot Overview in Pseudocode

## 1. The Big Picture

An operating system kernel has three fundamental responsibilities:

1. **CPU Management:** Who gets to run, and for how long?
2. **Memory Management:** Who gets which memory, and who can access it?
3. **Resource Management:** How do programs access files, devices, and other programs safely?

The kernel uses hardware mechanisms such as CPU privilege levels, interrupts, and the MMU to implement these responsibilities.

## 2. The Entire Kernel in Pseudocode

```python
# Conceptual OS pseudocode

def kernel_main():
    # Initialize hardware-dependent mechanisms
    setup_physical_memory_allocator()
    setup_virtual_memory()
    setup_trap_table()
    setup_timer()
    setup_devices()

    # Start the first user process
    create_init_process()

    # Begin CPU scheduling
    schedule()


def trap_handler(event):
    if event == TIMER:
        maybe_reschedule()

    elif event == SYSCALL:
        handle_syscall()

    elif event == PAGE_FAULT:
        handle_page_fault()

    elif event == DEVICE_INTERRUPT:
        wake_waiting_processes()

    else:
        handle_exception()


def handle_syscall(request):
    if request == FORK:
        create_child_process()

    elif request == EXEC:
        replace_process_program()

    elif request == MMAP:
        update_memory_mappings()

    elif request == READ:
        read_from_resource()

    elif request == WRITE:
        write_to_resource()


def schedule():
    next_process = choose_runnable_process()
    context_switch_to(next_process)
```

This is a conceptual architecture, not executable kernel code. Real kernels require additional initialization, permission checks, synchronization, process state management, and architecture-specific code.

---

## 3. Kernel Initialization

### `setup_physical_memory_allocator()`

**Problem:** Physical RAM is finite. The kernel must know which memory pages are available.

Responsibilities:

- Discover usable physical RAM.
- Track free and allocated physical pages.
- Allocate physical pages when needed.
- Reclaim pages when they are no longer used.

Conceptual implementation:

```python
free_pages = [2, 3, 4, 5]

def allocate_page():
    return free_pages.pop()

def free_page(page):
    free_pages.append(page)
```

**Hardware relationship:** Physical memory allocation is primarily software bookkeeping. The CPU does not provide a high-level `allocate_page()` instruction.

### `setup_virtual_memory()`

**Problem:** Each process needs an isolated address space, even though all processes share physical RAM.

Responsibilities:

- Create and maintain page tables.
- Map virtual pages to physical pages.
- Set read/write/execute and user/kernel permissions.
- Configure the CPU's page-table root.
- Handle changes to mappings and related TLB state.

Example:

```python
physical_page = allocate_page()

map_page(
    process="A",
    virtual_address=0x1000,
    physical_page=physical_page,
    permissions="USER_READ_WRITE"
)
```

Hardware building blocks:

| Hardware | Purpose |
|---|---|
| MMU | Translates virtual addresses into physical addresses |
| Page Tables | Store mappings and access permissions |
| Page Table Root Register | Identifies the active page-table hierarchy, e.g. x86 `CR3` |
| Page Fault Mechanism | Reports missing or forbidden mappings |
| TLB | Caches recent address translations |

**Key insight:** The kernel creates and modifies the mappings. The CPU/MMU enforces them during normal memory access.

### `setup_trap_table()`

**Problem:** The CPU needs to know which kernel handler to execute when a syscall, interrupt, or exception occurs.

Conceptual table:

| Event | Handler |
|---|---|
| Timer Interrupt | `handle_timer()` |
| System Call | `handle_syscall()` |
| Page Fault | `handle_page_fault()` |
| Device Interrupt | `handle_device_interrupt()` |

The kernel installs protected handler entry points through architecture-specific mechanisms.

For example, x86 uses the Interrupt Descriptor Table (IDT) for interrupt and exception delivery.

**Key insight:** The kernel installs handlers; CPU hardware transfers control to them when appropriate.

### `setup_timer()`

**Problem:** A user program might never voluntarily release the CPU.

```python
while True:
    pass
```

The kernel configures hardware timer interrupts to regain control periodically.

A timer interrupt may result in:

```python
def handle_timer():
    if time_slice_expired():
        maybe_reschedule()
```

**Key insight:** Hardware provides a way to interrupt execution; the OS decides whether to switch processes.

### `setup_devices()`

**Problem:** User programs need to interact with the outside world without directly controlling all hardware.

Examples:

- Console
- Disk
- Keyboard
- Network
- Display

The kernel initializes device drivers and exposes controlled interfaces to programs.

A minimal educational OS can initially support only a console and timer.

---

## 4. Trap Handling: How the Kernel Regains Control

A running user program does not continuously ask the kernel for permission to execute instructions.

It runs directly on the CPU until an event requires kernel intervention.

Three especially important events are:

| Event | Origin | Example |
|---|---|---|
| System Call | User program | `read()`, `fork()`, `mmap()` |
| Timer Interrupt | Hardware timer | Time slice expires |
| Page Fault | CPU memory access checks | Missing mapping or permission violation |

Conceptual execution flow:

```text
User Program
     |
     | Syscall / Interrupt / Exception
     v
CPU enters protected kernel handler
     |
     v
Kernel handles event
     |
     | Possibly schedules another process
     v
Return to User Mode
```

**Key insight:** The CPU enforces privilege transitions, while the kernel decides how to handle each event.

---

## 5. System Calls: What Services Do User Programs Need?

A user program can perform ordinary calculations and access memory that has already been mapped with appropriate permissions.

But it needs the kernel when it wants to manage protected or shared resources.

| Category | Fundamental Need | Typical Linux Syscalls |
|---|---|---|
| Process Management | Create, execute, terminate, and wait for programs | `fork`, `execve`, `exit`, `wait4` |
| Memory Management | Create or modify memory mappings | `mmap`, `munmap`, `mprotect`, `brk` |
| File / Device I/O | Open, read, write, and close resources | `openat`, `read`, `write`, `close` |
| IPC / Networking | Communicate with other processes or machines | `pipe2`, `socket`, `sendto`, `recvfrom` |
| Synchronization / Time | Wait, sleep, coordinate concurrent execution | `futex`, `epoll_wait`, `nanosleep` |
| Security / Control | Manage permissions, signals, and process attributes | `setuid`, `kill`, `rt_sigaction` |

### Important distinction: Memory allocation vs. system call

Creating a Python object:

```python
x = [1, 2, 3]
```

does not necessarily trigger a system call.

Python may allocate memory from an existing user-space memory pool.

When additional virtual memory is needed, its allocator may eventually request it using an OS interface such as `mmap()`.

A first access to an unmapped physical page may then trigger a page fault, which the kernel handles.

**System Call:** A program explicitly requests a service.

**Page Fault:** The CPU detects a memory access condition that requires kernel handling.

---

## 6. Process Creation: `fork()` and `exec()`

### Why does Linux use `fork()`?

From first principles, we could design:

```python
new_process("/bin/python", ["app.py"])
```

There is no hardware requirement that process creation must use `fork()`.

Unix instead separates two operations:

- **`fork()`**: Create a child process with an initially similar execution environment.
- **`exec()`**: Replace the current process's program with a different program.

Example:

```python
pid = fork()

if pid == 0:
    exec("/bin/python", ["app.py"])
else:
    wait(pid)
```

Execution flow:

```text
Shell (PID 100)
      |
      | fork()
      |
      +------------------+
      |                  |
Parent (PID 100)     Child (PID 101)
      |                  |
    wait()             exec()
                         |
                         v
                   Python (PID 101)
```

The child's PID remains the same after `exec()`.

### Why separate `fork()` and `exec()`?

Because the child can modify its environment before executing a different program.

For example:

```python
pid = fork()

if pid == 0:
    redirect_stdout("output.txt")
    exec("/bin/python", ["app.py"])
```

The new Python program inherits the redirected standard output.

Modern Linux typically uses Copy-on-Write to avoid eagerly duplicating all physical memory pages during `fork()`.

Parent/child relationships are OS abstractions, not CPU hardware requirements.

They help manage process creation, inheritance, exit status, and waiting.

Alternative interfaces such as `posix_spawn()` can combine process creation and program execution.

---

## 7. Scheduling: When Does the Kernel Switch Processes?

**Problem:** Multiple runnable programs must share a finite number of CPU cores.

The scheduler decides which runnable process or thread executes next.

Scheduling may occur because of:

1. **Timer Interrupt:** The current task's time slice expires.
2. **Blocking Operation:** The current task must wait for I/O or another event.
3. **Process Exit:** The current task terminates.
4. **Voluntary Yield:** A task explicitly gives up the CPU.

Conceptually:

```python
def schedule():
    next_process = choose_runnable_process()
    context_switch_to(next_process)
```

Real systems separate scheduler decisions from the precise point at which a context switch is safe.

Typical process states:

| State | Meaning |
|---|---|
| RUNNING | Currently executing |
| RUNNABLE | Ready and waiting for CPU time |
| BLOCKED / SLEEPING | Waiting for an event |
| ZOMBIE | Terminated, with exit information awaiting collection |

**Key insight:** A blocked process should not consume CPU time while it is waiting for an event.

---

## 8. The First User Process

Once the kernel has initialized the necessary mechanisms, it starts an initial user-space program.

On many modern Linux distributions, the main init system is `systemd`, running as PID 1.

Conceptually:

```text
CPU Reset
    ↓
Firmware / Bootloader
    ↓
Linux Kernel
    ↓
init / systemd (PID 1)
    ↓
System Services
    ↓
Login Manager
    ↓
Desktop Environment
    ↓
User Applications
```

This is a simplified boot sequence. Real Linux systems may begin with an initramfs `/init`, and not all processes descend directly from `systemd`.

Even when the user does nothing on the desktop, many user-space processes exist. Most are sleeping while waiting for events.

**Important:** `systemd` running as root does not mean it runs in CPU Kernel Mode.

Root is an OS-level identity; Ring 0 is a hardware privilege level.

---

## 9. Kernel vs. Hardware: Responsibility Summary

| Responsibility | Kernel Software | Hardware |
|---|---|---|
| Decide which process runs next | Yes | |
| Save and restore execution state | Yes, using CPU mechanisms | Provides registers and transition mechanisms |
| Decide which RAM pages are allocated | Yes | |
| Translate virtual addresses | Configures mappings | MMU performs translation |
| Enforce page permissions | Configures permissions | CPU/MMU enforces them |
| Define system call services | Yes | |
| Perform protected syscall entry | Provides handler | CPU performs privilege transition |
| Generate timer interrupts | Configures timer | Timer/interrupt hardware |
| Handle page faults | Yes | CPU detects faults |
| Implement pipes and files | Yes | Supplies underlying memory and devices |

---

## 10. Final Mental Model

The kernel can be understood as the **resource arbiter** of the computer.

### CPU

**Who gets to execute?**

- Scheduler
- Processes and threads
- Context switching
- Interrupt handling

### Memory

**Who gets which memory, and who may access it?**

- Physical memory allocator
- Virtual memory
- Page tables
- Page faults
- Memory protection

### Resources and Communication

**What may programs access, and how do they interact?**

- System calls
- Devices and I/O
- Files
- Pipes and sockets
- Synchronization
- Permissions

The relationship between the components is:

```text
Hardware
  |
  | Privilege Levels, Interrupts, MMU
  v
Kernel
  |
  | Scheduling, Memory Management, System Calls
  v
User Processes
  |
  | fork(), exec(), mmap(), read(), write(), ...
  v
Kernel Services
```

### The most important takeaway

**The kernel is not continuously executing to supervise every instruction.**

Instead:

1. The kernel configures hardware-enforced rules.
2. User programs execute directly on the CPU within those rules.
3. System calls, interrupts, and exceptions transfer control to the kernel when needed.
4. The kernel manages resources and may select another process.
5. CPU execution returns to user mode.

This is the fundamental architecture that makes modern multiprogramming operating systems possible.